In [ ]:
!pip install -qU ultralytics

## Initialize SAM Model

In [1]:
from ultralytics import SAM
import matplotlib.pyplot as plt

# load the model
model = SAM('sam2.1_b.pt')

# display model info
model.info()

Matplotlib is building the font cache; this may take a moment.


Model summary: 436 layers, 80,850,178 parameters, 80,850,178 gradients


(436, 80850178, 80850178, 0.0)

In [2]:
# url - https://ultralytics.com/images/bus.jpg

## Segment Image

In [2]:
# define bounding box regions
bboxes = [[55, 400, 230, 900]]

image_path = 'test_image.jpg'
results = model(image_path, bboxes=bboxes)


image 1/1 /Users/kaustubhraymane/Downloads/Deep-Learning-Projects/Image & Video Segmentation using SAM2.1/test_image.jpg: 1024x1024 1 0, 5593.0ms
Speed: 38.7ms preprocess, 5593.0ms inference, 14.1ms postprocess per image at shape (1, 3, 1024, 1024)


In [3]:
for result in results:
    result.show()

In [4]:
# define single points
points = [[350, 370]]
results = model(image_path, points=points, labels=[1])


image 1/1 /Users/kaustubhraymane/Downloads/Deep-Learning-Projects/Image & Video Segmentation using SAM2.1/test_image.jpg: 1024x1024 1 0, 5148.1ms
Speed: 34.9ms preprocess, 5148.1ms inference, 6.9ms postprocess per image at shape (1, 3, 1024, 1024)


In [5]:
for result in results:
    result.show()

In [6]:
# define multiple points
points = [[350, 370], [100, 650]]
results = model(image_path, points=points, labels=[1, 0])


image 1/1 /Users/kaustubhraymane/Downloads/Deep-Learning-Projects/Image & Video Segmentation using SAM2.1/test_image.jpg: 1024x1024 1 0, 4365.3ms
Speed: 30.9ms preprocess, 4365.3ms inference, 2.5ms postprocess per image at shape (1, 3, 1024, 1024)


In [7]:
for result in results:
    result.show()

## Extract BBox Image from the Original Image

In [8]:
import cv2
import torch
import numpy as np

In [9]:
result.boxes.xyxy

tensor([[ 17., 232., 800., 726.]])

In [ ]:
image = cv2.imread(image_path)

for i, result in enumerate(results):
    if hasattr(result, 'boxes') and result.boxes is not None:
        boxes = result.boxes.xyxy.cpu().numpy() if isinstance(result.boxes.xyxy, torch.Tensor) else np.array(result.boxes.xyxy)

        # iterate through the bounding boxes
        for j, box in enumerate(boxes):
            x1, y1, x2, y2 = map(int, box[:4])

            cropped_img = image[y1:y2, x1: x2]

            # show the image
            cv2.imshow(f"Cropped Image {i}_{j}", cropped_img)
            cv2.waitKey(0)

cv2.destroyAllWindows()

## Segment Video

In [ ]:

from ultralytics.models.sam import SAM2VideoPredictor

overrides = dict(
    conf=0.25,
    task="segment",
    mode="predict",
    imgsz=512,
    model="sam2.1_b.pt",
    device="cpu",
    half=False,
    save=True,
    vid_stride=3
)

predictor = SAM2VideoPredictor(overrides=overrides)

In [ ]:
video_path = 'test_video.mp4'

results = predictor(source=video_path, points=[900, 820], labels=[1])